In [ ]:
# !pip install pikepdf

In [ ]:
import os, itertools
from concurrent.futures import ProcessPoolExecutor
from worker_module import compute_task, worker_task
from tqdm import tqdm

CHECKPOINT_FILE = "last_processed_line.txt"
FAILURES_FILE = "failed_lines.txt"
INPUT_FILE = "words.txt"
CHECKPOINT_INTERVAL =  chunksize = 100000

def chunked_iterable(iterable, size):
    """Slices an iterable into batches of `size` items."""
    while True:
        chunk = list(itertools.islice(iterable, size))
        if not chunk:
            break
        yield chunk
        

def get_last_checkpoint(filepath: str) -> int:
    """Reads the last successfully processed line number (0-indexed)."""
    if os.path.exists(filepath):
        try:
            with open(filepath, "r") as f:
                content = f.read().strip()
                if content:
                    return int(content)
        except (ValueError, IOError):
            pass
    return -1


def word_line_generator(filepath: str, start_line: int):
    """
    Yields (line_number, word) line-by-line from the input file,
    skipping up to start_line.
    """
    with open(filepath, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f):
            if line_num <= start_line:
                continue
            word = line.strip()
            if word:  # Skip empty lines
                yield line_num, word


# # Placeholder for your actual task logic
# def worker_task(item):
#     line_num, word = item
#     # Example: replace with your actual compute logic
#     # result = compute_task(word)
#     result = compute_task(word)
#     return line_num, word, result

In [ ]:
# if __name__ == "__main__":
# 1. Resume from last saved checkpoint
last_completed_line = get_last_checkpoint(CHECKPOINT_FILE)
if last_completed_line >= 0:
    print(f"Resuming from line: {last_completed_line + 1}")
else:
    print("Starting from beginning.")

# 2. Estimate total lines for tqdm
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    total_lines = sum(1 for _ in f)
remaining_tasks = max(0, total_lines - (last_completed_line + 1))

# 3. Create lazy task stream
task_stream = word_line_generator(INPUT_FILE, last_completed_line)

num_cores = os.cpu_count() or 4
max_workers = max(1, int(num_cores * 0.5))



# 4. Process lazily with bounded chunksize for reliable checkpoint tracking
with ProcessPoolExecutor(max_workers=max_workers) as executor, \
    tqdm( total=remaining_tasks, unit="word" ) as pbar:
    # Lower chunksize ensures results yield back promptly for checkpointing
    for chunk in chunked_iterable(task_stream, CHECKPOINT_INTERVAL):
        # 2. Process this chunk across workers
        # chunksize=50 to 100 distributes this batch evenly across workers
        results = executor.map(worker_task, chunk, chunksize=100)

        last_line_in_chunk = None
        for line_num, word, result in results:
            # if not result:
            #     fail_f.write(f"{line_num}\t{word}\n")
            last_line_in_chunk = line_num
            # pbar.update(1)
        pbar.update(chunksize)
        if last_line_in_chunk is not None:
            with open(CHECKPOINT_FILE, "w", encoding="utf-8") as cp_f:
                cp_f.write(str(last_line_in_chunk))
            # fail_f.flush()
            # results_iterator = executor.map(worker_task, task_stream, chunksize=chunksize)
        
        # processed_counter = 0

    # with open(FAILURES_FILE, "a", encoding="utf-8") as fail_f:
    # for line_num, word, result in tqdm(
    #         results_iterator, total=remaining_tasks, unit="word"
    #     ):
    #         # # Check if task produced no result
    #         # if not result:
    #         #     fail_f.write(f"{line_num}\t{word}\n")
    #         #     fail_f.flush()

    #     # processed_counter += 1

    #     # Update progress checkpoint every 1,000 processed rows
    #     if line_num % CHECKPOINT_INTERVAL == 0:
    #         with open(CHECKPOINT_FILE, "w", encoding="utf-8") as cp_f:
    #             cp_f.write(str(line_num))

    # # Final checkpoint update at the end
    # with open(CHECKPOINT_FILE, "w", encoding="utf-8") as cp_f:
    #     cp_f.write(str(total_lines - 1))
# 175625960
# 175325960
# 174525960
# 2520000

In [ ]:
sys.exit()

In [ ]:
with ProcessPoolExecutor(max_workers=max_workers) as executor:
    with open(FAILURES_FILE, "a", encoding="utf-8") as fail_f, tqdm(
        total=remaining_tasks, unit="word"
    ) as pbar:

        # 1. Iterate chunk-by-chunk (1,000 lines per batch)
        for chunk in chunked_iterable(task_stream, CHECKPOINT_INTERVAL):
            # 2. Process this chunk across workers
            # chunksize=50 to 100 distributes this batch evenly across workers
            results = executor.map(worker_task, chunk, chunksize=100)

            last_line_in_chunk = None

            # 3. Consume results of the current chunk
            for line_num, word, result in results:
                if not result:
                    fail_f.write(f"{line_num}\t{word}\n")
                last_line_in_chunk = line_num
                pbar.update(1)

            fail_f.flush()

            # 4. Reached boundary: write checkpoint BEFORE fetching the next chunk
            if last_line_in_chunk is not None:
                with open(CHECKPOINT_FILE, "w", encoding="utf-8") as cp_f:
                    cp_f.write(str(last_line_in_chunk))

In [ ]:
from pprint import pprint
from tqdm import tqdm
import itertools, string , os ,time
#, pikepdf
from concurrent.futures import ProcessPoolExecutor
from worker_module import compute_task

In [ ]:
dates = [ f"{i:02d}" for i in range(1, 32) ]
months = [ f"{i:02d}" for i in range(1, 13) ]
date_month_combos = ["".join(d_combo) for d_combo in itertools.product(dates, months)]
# [list(range(1, 32)), list(range(1, 13))]
# print(dates)
# print(months)
# print(date_month_combos)

In [ ]:
letters_pool = string.ascii_lowercase 
digits_pool = string.digits

# print("🔑 Attack started. Testing combinations (1 to 4 letters + 4 digits)...")

# 2. Outer loop: Letter length variations (1 letter up to 4 letters)
# dicty = {}
for length in range(4, 0, -1):  # From 4 down to 1
    # print(f"🔄 Testing patterns with {length} letter(s) and 4 digits...")
    
    # Generator for the letter combinations
    letter_combos = ["".join(l_combo) for l_combo in itertools.product(letters_pool, repeat=length)]
    
    password_combos = ["".join(password_combo) for password_combo in itertools.product(letter_combos, date_month_combos)]
    with open("output.txt", "a") as file:
        for item in password_combos:
            # Remove all spaces and write with a newline character
            cleaned_item = str(item).strip()
            file.write(f"{cleaned_item}\n")
    # break
#     dicty[length] = password_combos
#     print()
# pprint(dicty)

# my_list = ["apple ", " ba na na ", "orange", "grape fruit"]


In [ ]:
# 1. Generator yielding combinations on the fly
def generate_word_combinations(letters_pool   , date_month_combos):
    for length in range(1, 5):
        for l_combo in itertools.product(letters_pool, repeat=length):
            prefix = "".join(l_combo)
            for suffix in date_month_combos:
                yield prefix + suffix


# print(f"Available CPU cores: {num_cores}")
letters_pool = string.ascii_lowercase
letters_len = len(letters_pool)
combos_len = len(date_month_combos)
total_tasks = sum(letters_len ** l for l in range(1, 5)) * combos_len
print(f"Total tasks to process: {total_tasks}")
# Instantiate generator (consumes almost 0 RAM)
task_generator = generate_word_combinations(letters_pool, date_month_combos)

In [ ]:
num_cores = os.cpu_count()
with ProcessPoolExecutor(max_workers=int(num_cores*.75)) as executor:
    # Pass the generator directly with an optimized chunksize
    results_iterator = executor.map(
        compute_task, 
        task_generator, 
        chunksize=100000
    )

    # 3. Consume the results lazily as they complete
    # Pass the pre-calculated total so tqdm can track % and ETA
    for result in tqdm(results_iterator, total=total_tasks, unit="task"):
        # Process results one by one (e.g., write to disk, aggregate, etc.)
        pass


In [ ]:
# letters_pool = string.ascii_lowercase 
# digits_pool = string.digits

# print("🔑 Attack started. Testing combinations (1 to 4 letters + 4 digits)...")

# 2. Outer loop: Letter length variations (1 letter up to 4 letters)
# dicty = {}
# for length in range(4, 0, -1):  # From 4 down to 1
#     # print(f"🔄 Testing patterns with {length} letter(s) and 4 digits...")
    
#     # Generator for the letter combinations
#     letter_combos = ["".join(l_combo) for l_combo in itertools.product(letters_pool, repeat=length)]
    
#     password_combos = ["".join(password_combo) for password_combo in itertools.product(letter_combos, date_month_combos)]
#     dicty[length] = password_combos
    # print()
# pprint(dicty)

In [ ]:
# for length, passwords in dicty.items():
#     print(length, len(passwords))

In [ ]:
# A CPU-bound worker function
def crack_pdf_pattern(password):
    try:
        # Attempt to decrypt
        with pikepdf.open(pdf_path = "check.pdf", password=password):
            print(f"\n✅ Success! Password found: {password}")
            return password
    except pikepdf.PasswordError:
        # continue  # Wrong password, keep trying
        return None
    except Exception as e:
        print(f"\n⚠️ Unexpected error: {e}")
        return None
                    
    # print("\n❌ Password not found within the specified pattern boundaries.")
    # return None

In [ ]:
cores = os.cpu_count()
print(f"Running across {cores} CPU cores...")

start_time = time.time()

# ProcessPoolExecutor defaults to max_workers=os.cpu_count()
# for key in dicty:
passwords = dicty[1]
with ProcessPoolExecutor(max_workers=int(cores/2)) as executor:
    results = list(executor.map(crack_pdf_pattern, passwords))

elapsed = time.time() - start_time
print(f"Completed in {elapsed:.2f} seconds.")

In [ ]:
# 1. Generator yielding combinations on the fly
def generate_word_combinations(letters_pool   , date_month_combos):
    for length in range(4, 0, -1):
        for l_combo in itertools.product(letters_pool, repeat=length):
            prefix = "".join(l_combo)
            for suffix in date_month_combos:
                yield prefix + suffix

# 2. Worker function executed across CPU processes
def compute_heavy_task(item):
    try:
        # Attempt to decrypt
        with pikepdf.open(pdf_path = "check.pdf", password=item):
            print(f"\n✅ Success! Password found: {item}")
            return item
    except pikepdf.PasswordError:
        # continue  # Wrong password, keep trying
        return None
    except Exception as e:
        print(f"\n⚠️ Unexpected error: {e}")
        return None

# # cores = os.cpu_count() or 2
# letters_pool = string.ascii_lowercase
# date_month_combos = ["0101", "0202", "1231"]

# Instantiate generator (consumes almost 0 RAM)
task_generator = generate_word_combinations(letters_pool, date_month_combos)

with ProcessPoolExecutor(max_workers=int(os.cpu_count()/2)) as executor:
    # Pass the generator directly with an optimized chunksize
    results_iterator = executor.map(
        compute_heavy_task, 
        task_generator, 
        chunksize=50
    )

    # 3. Consume the results lazily as they complete
    for result in results_iterator:
        # Process results one by one (e.g., write to disk, aggregate, etc.)
        pass

In [ ]:
!ls

In [ ]:
import os
import string
import itertools
from concurrent.futures import ProcessPoolExecutor

# 1. Generator yielding combinations on the fly
def generate_word_combinations(letters_pool, date_month_combos):
    for length in range(4, 0, -1):
        for l_combo in itertools.product(letters_pool, repeat=length):
            prefix = "".join(l_combo)
            for suffix in date_month_combos:
                yield prefix + suffix

# 2. Worker function executed across CPU processes
def compute_heavy_task(item):
    # Perform task on item
    # Keep return value minimal to avoid memory consumption
    return len(item)

if __name__ == "__main__":
    cores = os.cpu_count() or 2
    letters_pool = string.ascii_lowercase
    date_month_combos = ["0101", "0202", "1231"]

    # Instantiate generator (consumes almost 0 RAM)
    task_generator = generate_word_combinations(letters_pool, date_month_combos)

    with ProcessPoolExecutor(max_workers=cores) as executor:
        # Pass the generator directly with an optimized chunksize
        results_iterator = executor.map(
            compute_heavy_task, 
            task_generator, 
            chunksize=5000
        )

        # 3. Consume the results lazily as they complete
        for result in results_iterator:
            # Process results one by one (e.g., write to disk, aggregate, etc.)
            pass

In [ ]:
print("hello World")

In [ ]:
# save as test_cec.py
import subprocess
import time

commands = [
    {"desc": "Broadcast ON (F0:72:01)", "cmd": "cmd=0xF0,payload=0x72:0x01", "to": "0"},
    {"desc": "Direct ON to soundbar (50:72:01)", "cmd": "cmd=0x50,payload=0x72:0x01", "to": "5"},
    {"desc": "Broadcast OFF (F0:72:00)", "cmd": "cmd=0xF0,payload=0x72:0x00", "to": "0"},
    {"desc": "Mute (50:44:43)", "cmd": "cmd=0x50,payload=0x44:0x43", "to": "5"},
    {"desc": "Volume up (50:44:41)", "cmd": "cmd=0x50,payload=0x44:0x41", "to": "5"},
    {"desc": "Request handover (45:70)", "cmd": "cmd=0x45,payload=0x70", "to": "5"},
]

for c in commands:
    print(f"\nSending: {c['desc']}")
    args = ["cec-ctl", "-d", "/dev/cec0", "--custom-command", c["cmd"]]
    if "to" in c:
        args.extend(["--to", c["to"]])
    subprocess.run(args)
    time.sleep(3)  # time to hear/see change